# HW10

Author: Evan Whitfield

Course: ST 554 - Spring 2026

Date Editted: 4/21/2026

Purpose: Answer HW10

## Part 1 - Creating Stream Data Using `rate`



Setup a data stream using the `rate` format.

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sqrt, col

spark = SparkSession.builder.appName("HW10Part1").getOrCreate()

In [17]:
rate_df = spark.readStream.format("rate").load()

Prior to starting the stream, set up a sequence of actions using appropriate functions from pyspark.sql.functions
that uses the rate data to

• find the square root of the rate `value`

• find mod 4 of the rate `value`

In [30]:
modified_df = rate_df \
    .withColumn("sqrt_value", sqrt(col("value"))) \
    .withColumn("mod4_value", col("value") % 4)

To output this, create a writeStream that writes to `memory` (format(`memory`)). Give the query a name
(queryName("...")) and start it!

In [37]:
query = modified_df.writeStream \
    .format("memory") \
    .queryName("rate_table") \
    .outputMode("append") \
    .start()

26/04/21 22:15:55 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c44a362c-1d16-43ed-a46e-caf6f91ab83b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 22:15:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [42]:
query.stop()

26/04/21 22:16:31 WARN DAGScheduler: Failed to cancel job group f2bd00c5-8fd2-4df7-b618-8dcdc8bfd456. Cannot find active jobs for it.
26/04/21 22:16:31 WARN DAGScheduler: Failed to cancel job group f2bd00c5-8fd2-4df7-b618-8dcdc8bfd456. Cannot find active jobs for it.


Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the query
name (spark.sql("select * from you_table_name").show()).

In [43]:
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-21 22:15:...|    0|               0.0|         0|
|2026-04-21 22:15:...|    1|               1.0|         1|
|2026-04-21 22:15:...|    2|1.4142135623730951|         2|
|2026-04-21 22:15:...|    3|1.7320508075688772|         3|
|2026-04-21 22:15:...|    4|               2.0|         0|
|2026-04-21 22:16:...|    5|  2.23606797749979|         1|
|2026-04-21 22:16:...|    6| 2.449489742783178|         2|
|2026-04-21 22:16:...|    7|2.6457513110645907|         3|
|2026-04-21 22:16:...|    8|2.8284271247461903|         0|
|2026-04-21 22:16:...|    9|               3.0|         1|
|2026-04-21 22:16:...|   10|3.1622776601683795|         2|
|2026-04-21 22:16:...|   11|   3.3166247903554|         3|
|2026-04-21 22:16:...|   12|3.4641016151377544|         0|
|2026-04-21 22:16:...|   13| 3.605551275463989|         

In [46]:
spark.stop()

## Part 2 - Using data from a CSV with a Pipeline

In [52]:
from pyspark.sql.functions import *
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline

bikeDetails_for_fit.csv should be read in as a spark (SQL) data frame.

In [53]:
spark_bike = SparkSession.builder.appName("BikePipeline").getOrCreate()

In [70]:
training_df = spark_bike.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)
training_df.show(20)

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
|    Honda CB Twister|        18000|2010| Individual|1st owner|    60000|            53857|
|Honda CB Hornet 160R|        78500|2018| Individual|1st owner|    17000|            87719|
|Royal Enfield Bul...|       180000|2008| Individual|2nd owner|    39000|       

Use an SQLTransformer to do some log transforms, rename a variable, and create a dummy variable from categorical variable.

In [72]:
transformer = SQLTransformer(
    statement="""SELECT log(selling_price) AS label,year, log(km_driven) AS log_km_driven,
            CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner FROM __THIS__
    """
)

Use a VectorAssembler to create a features column. The features column should include the year,
log_km_driven, and one_owner variables.

In [67]:
vector_assembler = VectorAssembler(inputCols=["year", "log_km_driven", "one_owner"],outputCol="features")

Create a Pipeline with the two steps above (SQLTransformer then VectorAssembler)

In [73]:
pipeline = Pipeline(stages=[transformer, vector_assembler])

Fit this pipeline to the SQL data frame and save this as an object.

In [74]:
fitted_pipeline = pipeline.fit(training_df)

Now we want to set up a read stream to look for csv files placed into a folder (the five bikeDetails_add*.csv files). When a csv comes in, we want to transform it using the fitted pipeline’s .transform() method! 

In [75]:
schema = training_df.schema

In [76]:
stream_df = spark_bike.readStream \
         .option("header", True) \
         .schema(schema) \
         .csv("Bike_Data")

In [78]:
new_stream = fitted_pipeline.transform(stream_df)

We’ll just write the output to the ‘console’ using the ‘append’ mode.

In [79]:
query = new_stream.writeStream.format("console").outputMode("append").start()

26/04/21 22:43:18 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-a531aaaf-7ef2-40d9-b724-1f9af0bfdad1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 22:43:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

Once that is all set up, make sure your folder you are looking for the .csv files is empty. Then start the query. Place the other files into the folder one at a time. You should see the output appear below your query. Once you’ve done all five files, stop the query!

In [80]:
query.stop()

26/04/21 22:46:01 WARN DAGScheduler: Failed to cancel job group c05b3d0e-71ee-4fa2-85d2-25eee87513da. Cannot find active jobs for it.
26/04/21 22:46:01 WARN DAGScheduler: Failed to cancel job group c05b3d0e-71ee-4fa2-85d2-25eee87513da. Cannot find active jobs for it.


All finished! Now we should be able to complete any predictions on the new csv files!